<a href="https://colab.research.google.com/github/zhiyuguo-innis/MSSP607/blob/main/zhiyuguo-innis/MSSP607/Modules/Week_6/PARTICIPATION_ACTIVITY_Week_6_Write_a_lamda_function_for_a_dataframe.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# MSSP 607 — Participation Activity, PARTICIPATION ACTIVITY Week 6: Write a lamda function for a dataframe

**Name:** Zhiyu Guo

### Reading notes

**Lambda expressions** ([Python reference §6.14](https://docs.python.org/3/reference/expressions.html#lambda)).
`lambda parameters: expression` creates an anonymous function. It behaves exactly like a function
defined with `def` that returns that one expression. The body must be a single *expression* —
no statements such as `if` blocks, `for` loops, assignments, or `return`. Lambdas are most useful
for short, throwaway functions passed to other functions (`sorted(key=...)`, `map`, `filter`,
and in pandas `apply`, `assign`, `pipe`).

**PEP 312 — Simple Implicit Lambda** ([PEP 312](https://peps.python.org/pep-0312/)).
Proposed letting you drop the `lambda` keyword in some contexts (e.g. writing `:x + 1` instead of
`lambda: x + 1`) to make passing small functions less verbose. The PEP was **deferred and never
adopted**, so the explicit `lambda` keyword is still required in all current Python versions.


### 1. Load the data
In Colab, upload `student-mat.csv` from Canvas (or mount Drive). The UCI file uses `;` as its separator, so `sep=None` lets pandas detect it automatically.

In [2]:
import pandas as pd

url = ("https://raw.githubusercontent.com/zhiyuguo-innis/MSSP607/main/"
       "zhiyuguo-innis/MSSP607/Modules/Week_6/Student%20Performance%20Data.csv")

students = pd.read_csv(url)
print(students.shape)
students.head()


(1000, 10)


,gender,race/ethnicity,parental level of education,lunch,math score,reading score,writing score,total score,test preparation course,admission prospect
0,female,group B,bachelor's degree,standard,72,72,74,218,none,medium
1,female,group C,some college,standard,69,90,88,247,completed,high
2,female,group B,master's degree,standard,90,95,93,278,none,high
3,male,group A,associate's degree,free/reduced,47,57,44,148,none,low
4,male,group C,some college,standard,76,78,75,229,none,high


In [3]:
students[[ "math score", "reading score", "writing score", "total score"]].describe()

,math score,reading score,writing score,total score
count,1000.00000,1000.000000,1000.000000,1000.000000
mean,66.08900,69.169000,68.054000,203.312000
std,15.16308,14.600192,15.195657,42.771978
min,0.00000,17.000000,10.000000,27.000000
25%,57.00000,59.000000,57.750000,175.000000
50%,66.00000,70.000000,69.000000,205.000000
75%,77.00000,79.000000,79.000000,233.000000
max,100.00000,100.000000,100.000000,300.000000


### 2. A lambda that takes the DataFrame and returns another DataFrame

`score_report` takes the full student DataFrame and returns a **new** DataFrame that:
- adds `avg_score`, the mean of the math, reading and writing scores (each on a 0–100 scale),
- adds `all_passed`, which is True when the student scored at least 60 on all three tests (60 is a pass mark I chose for illustration, since the dataset has no official one),
- keeps a smaller set of relevant columns, sorted from highest to lowest average score.

Because `assign` returns a copy, the original `students` DataFrame is not modified.

In [4]:
scores = ["math score", "reading score", "writing score"]

score_report = lambda df: (
    df.assign(
        avg_score=df[scores].mean(axis=1).round(2),
        all_passed=(df[scores] >= 60).all(axis=1),
    )
    [["gender", "test preparation course", "lunch", *scores, "avg_score", "all_passed"]]
    .sort_values("avg_score", ascending=False)
    .reset_index(drop=True)
)

report = score_report(students)
print(type(report))
print(report.shape)
report.head(10)

<class 'pandas.core.frame.DataFrame'>
(1000, 8)


,gender,test preparation course,lunch,math score,reading score,writing score,avg_score,all_passed
0,male,completed,standard,100,100,100,100.00,True
1,female,none,standard,100,100,100,100.00,True
2,female,none,standard,100,100,100,100.00,True
3,female,completed,standard,99,100,100,99.67,True
4,female,none,standard,98,100,99,99.00,True
5,female,completed,standard,97,100,100,99.00,True
6,female,completed,standard,96,100,100,98.67,True
7,male,completed,standard,100,97,99,98.67,True
8,female,completed,standard,94,99,100,97.67,True
9,female,completed,free/reduced,93,100,100,97.67,True


In [5]:
print(students.shape)
print("avg_grade" in students.columns)

(1000, 10)
False


### 3. A second lambda: a summary DataFrame
`prep_summary` takes the student DataFrame and returns a grouped summary of average scores and pass rates by test preparation course and gender. It reuses `score_report`, showing that lambdas can be composed like any other function.

In [8]:
prep_summary = lambda df: (
    score_report(df)
    .groupby(["test preparation course", "gender"])
    .agg(n_students=("avg_score", "size"),
         mean_avg_score=("avg_score", "mean"),
         pass_rate=("all_passed", "mean"))
    .round(2)
    .reset_index()
)

prep_summary(students)

,test preparation course,gender,n_students,mean_avg_score,pass_rate
0,completed,female,184,74.46,0.70
1,completed,male,174,70.78,0.74
2,none,female,334,66.88,0.57
3,none,male,308,63.04,0.51


### 4. Lambdas inside pandas methods
Lambdas are often passed *into* pandas methods rather than called on their own. Here a lambda inside `apply` converts each math score to a letter grade, and a lambda inside `loc` selects the students who failed math.

In [9]:
letter = lambda s: "A" if s >= 90 else "B" if s >= 80 else "C" if s >= 70 else "D" if s >= 60 else "F"

with_grades = students.assign(math_grade=students["math score"].apply(letter))
print(with_grades["math_grade"].value_counts().sort_index())

# loc accepts a callable: the lambda receives the DataFrame and returns a boolean mask
failed_math = with_grades.loc[lambda df: df["math_grade"] == "F",
                              ["gender", "test preparation course", "math score", "math_grade"]]
print(failed_math.shape)
failed_math.head()

math_grade
A     58
B    135
C    216
D    268
F    323
Name: count, dtype: int64
(323, 4)


,gender,test preparation course,math score,math_grade
3,male,none,47,F
7,male,none,40,F
9,female,none,38,F
10,male,none,58,F
11,male,none,40,F


**Summary** This notebook practices Python lambda functions on the Student Performance Data Set (1,000 students; math, reading and writing scores), loaded directly from my MSSP607 GitHub repository. The main lambda, `score_report`, takes the student DataFrame and returns a new DataFrame. It adds each student's average score and a flag for passing all three tests (≥ 60), keeps the key columns, and sorts by average score, leaving the original data unchanged. A second lambda, `prep_summary`, builds on the first to compare groups. Students who completed the test preparation course averaged about 71–74, versus 63–67 for those who did not, and also had higher pass rates. The notebook also uses lambdas inside `apply` (converting math scores to letter grades; 323 students received an F) and inside `loc` (filtering rows). It closes with notes on lambda syntax and on PEP 312, a proposal to drop the `lambda` keyword that was never adopted.